# 机器学习概述（上）：五要素与线性/多项式回归

**机器学习**（Machine Learning，ML）就是让计算机从数据中进行自动学习，得到某种知识（或规律）。在实践中，机器学习通常指一类学习问题以及解决这类问题的方法，即如何从观测数据（样本）中寻找规律，并利用学习到的规律（模型）对未知或无法观测的数据进行预测。

本节按机器学习实践的**五要素**组织：数据、模型、学习准则、优化算法、评价指标。先用线性回归把五要素串成最小 pipeline，再用多项式回归直观看到**模型容量**对欠拟合 / 过拟合的影响，最后用 $\ell_2$ 正则压住过拟合。


## 0. 机器学习实践五要素

要通过机器学习来解决一个特定任务，我们需要准备 5 个方面的要素：

1. **数据**：收集任务相关的数据集来进行模型训练和测试。数据集可分为训练集、验证集和测试集。
2. **模型**：实现从样本特征到样本标签的映射，通常为带参数的函数。机器学习的目标就是学习一组最优的参数，使得模型的预测最"准确"。
3. **学习准则**：模型优化的目标，通常为训练集上的平均损失和正则化项的加权组合。有了学习准则之后，机器学习问题就转化为最优化问题。
4. **优化算法**：寻找最优参数的算法，通常为数学优化中的经典算法，比如梯度下降法、最小二乘法等。
5. **评价指标**：评价学习到的机器学习模型的性能，即如何衡量模型预测的"准确"程度。

从流程角度看，机器学习实践可以分为 4 个阶段：

- **模型准备**：准备数据、待学习的模型、损失函数、优化算法和评价指标。
- **模型训练**：基于学习准则和优化算法，使用训练集与验证集学习模型参数。
- **模型评价**：使用训练好的模型在测试集上用评价指标进行评价。
- **模型预测**：使用训练好的模型预测新样本的标签。

### 五要素 ↔ PyTorch 对照

| 要素 | 角色 | PyTorch 对应物 |
|---|---|---|
| 数据 | 训练 / 验证 / 测试集 | `torch.utils.data.Dataset`、`DataLoader` |
| 模型 | $f(x;\theta)$ | `nn.Module`（如 `nn.Linear`） |
| 学习准则 | 损失函数 + 正则项 | `nn.MSELoss`、优化器的 `weight_decay` |
| 优化算法 | 更新 $\theta$ | `torch.optim.SGD`、`torch.optim.Adam` 等 |
| 评价指标 | 度量模型效果 | 自定义函数 / `torchmetrics` |


## 1. 用线性回归把五要素串起来

**回归**（Regression）是一类典型的监督机器学习任务，对样本的输入和输出之间关系进行建模分析，其输出通常是一个连续值，比如房屋价格预测、电影票房预测等。**线性回归**（Linear Regression）是指一类利用线性函数来对输入和输出之间关系进行建模的回归任务。

在线性回归中，样本输入是特征向量 $\boldsymbol{x}\in\mathbb{R}^D$，输出是连续值标签 $y\in\mathbb{R}$。线性回归的模型定义为

$$ f(\boldsymbol{x};\boldsymbol{w},b)=\boldsymbol{w}^\top\boldsymbol{x}+b, $$

其中权重向量 $\boldsymbol{w}\in\mathbb{R}^D$ 和偏置 $b\in\mathbb{R}$ 都是可学习的参数。

实践中为了提高预测样本的效率，我们通常会把 $N$ 个样本归为一组进行成批预测，这样可以更好地利用 GPU 的并行计算能力。批量形式可以写为

$$ \boldsymbol{y}=\boldsymbol{X}\boldsymbol{w}+b, $$

其中 $\boldsymbol{X}\in\mathbb{R}^{N\times D}$ 为 $N$ 个样本的特征矩阵，$\boldsymbol{y}\in\mathbb{R}^N$ 为预测值组成的列向量。

> **形状约定**：为了和代码实现一致，我们使用"样本数量 × 特征维度"的张量来表示一组样本——样本矩阵 $\boldsymbol{X}$ 由 $N$ 个**行向量** $\boldsymbol{x}^\top$ 组成。这与《神经网络与深度学习》理论书中以 $\boldsymbol{x}$ 为列向量的写法刚好为转置关系。


### 1.1 数据：构造 ToyLinear150

我们构造一个用于线性回归的简单小规模数据集 **ToyLinear150**：先指定一个真实函数 $y = 1.2 x + 0.5$，然后加上高斯噪声，采样 150 个点。其中训练集 100 个，测试集 50 个。

数据划分时要权衡两个因素：更多的训练数据会降低参数估计的方差，从而得到更可信的模型；更多的测试数据会降低模型评价的方差，从而得到更可信的模型评价。一般可以按 7:3 或 8:2 划分训练集和测试集，再从训练集中按同样比例划出验证集。


In [ ]:
import math
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(0)


def linear_data(n, w=1.2, b=0.5, noise=2.0, interval=(-10, 10)):
    """按 y = w x + b + 高斯噪声 采样 n 个点。"""
    x = torch.empty(n, 1).uniform_(*interval)
    y = w * x + b + noise * torch.randn_like(x)
    return x, y


# 训练集 100 个，测试集 50 个；额外采一组 5000 个的"放大版"，用来后面对比样本量的影响
X_train, y_train = linear_data(100)
X_test,  y_test  = linear_data(50)
X_large, y_large = linear_data(5000)

# 可视化 ToyLinear150
xs = torch.linspace(-10, 10, 200).unsqueeze(1)
plt.scatter(X_train.numpy(), y_train.numpy(), s=15, alpha=0.6, label='train (100)')
plt.scatter(X_test.numpy(),  y_test.numpy(),  s=15, alpha=0.6, label='test (50)')
plt.plot(xs.numpy(), (1.2*xs + 0.5).numpy(), 'k-', lw=1, label='underlying')
plt.legend(); plt.xlabel('x'); plt.ylabel('y'); plt.title('ToyLinear150'); plt.show()


### 1.2 模型：基于 `Op` 类的线性算子

PyTorch 已经内置了 `nn.Linear`，但为了更深入地理解算子是如何工作的，本书也定义一个继承自第 1 章 `Op` 接口的 `Linear` 算子。它把权重和偏置作为成员变量 `params` 持有，便于后面手写最小二乘法直接赋值。


In [ ]:
class Op(object):
    """第 1 章定义的算子基类：要求子类实现 forward / backward。"""

    def __init__(self):
        pass

    def __call__(self, inputs):
        return self.forward(inputs)

    def forward(self, inputs):
        raise NotImplementedError

    def backward(self, outputs_grads):
        raise NotImplementedError


class Linear(Op):
    """y = X w + b 的线性算子（行向量样本）。"""

    def __init__(self, input_size):
        self.input_size = input_size
        # 模型参数（不参与 autograd，由最小二乘法直接赋值）
        self.params = {
            'w': torch.randn(input_size, 1),
            'b': torch.zeros(1),
        }

    def __call__(self, X):
        return self.forward(X)

    def forward(self, X):
        # X: [N, D]   ->   y_pred: [N, 1]
        assert X.shape[1] == self.input_size
        return X @ self.params['w'] + self.params['b']


# 简单测试：2 个样本、3 维特征
torch.manual_seed(0)
model_demo = Linear(input_size=3)
X_demo = torch.randn(2, 3)
print('y_pred:\n', model_demo(X_demo))


### 1.3 学习准则：均方误差

回归任务是对**连续值**的预测，希望模型能根据数据特征输出一个连续值作为预测值。因此回归任务中常用的学习准则是**均方误差**（Mean Squared Error，MSE）。

令 $\boldsymbol{y}\in\mathbb{R}^N$、$\hat{\boldsymbol{y}}\in\mathbb{R}^N$ 分别为 $N$ 个样本的真实标签和预测标签，均方误差定义为

$$ \mathcal{L}(\boldsymbol{y},\hat{\boldsymbol{y}})=\frac{1}{2N}\|\hat{\boldsymbol{y}}-\boldsymbol{y}\|^2=\frac{1}{2N}\|\boldsymbol{X}\boldsymbol{w}+\boldsymbol{b}-\boldsymbol{y}\|^2, $$

其中 $\boldsymbol{b}$ 为所有元素均为 $b$ 的 $N$ 维向量。

> 经验风险与结构风险：在训练集 $\mathcal{D}$ 上的平均损失称为**经验风险**（Empirical Risk）。当模型比较复杂或训练数据较少时，单纯最小化经验风险得到的模型容易在测试集上效果差——即**过拟合**（Overfitting）。为了缓解过拟合，通常会在经验损失上加一个正则化项（比如 $\ell_1$ 或 $\ell_2$ 范数），得到**结构风险**（Structure Risk）。


In [ ]:
def mean_squared_error(y_true, y_pred):
    """逐元素均方误差。"""
    assert y_true.shape[0] == y_pred.shape[0]
    return ((y_true - y_pred) ** 2).mean()


# 构造一个简单样例进行测试: [N, 1], N=2
y_true_t = torch.tensor([[-0.2], [4.9]])
y_pred_t = torch.tensor([[ 1.3], [2.5]])
print('error:', mean_squared_error(y_true_t, y_pred_t).item())


### 1.4 优化算法：最小二乘法与梯度法

给定学习准则之后，机器学习问题就转化为优化问题。可以利用已知的优化算法来求最优的模型参数。对于线性回归 + 均方误差，损失关于 $(\boldsymbol{w},b)$ 是凸函数，可以直接令偏导数等于 $0$ 得到**解析解**。这种方式叫**最小二乘法**（Least Square Method，LSM）。

对均方误差关于 $b$ 求偏导并令其为 $0$，可以解出

$$ b^* = \bar y - \bar{\boldsymbol{x}}^\top \boldsymbol{w}, $$

其中 $\bar y$ 和 $\bar{\boldsymbol{x}}$ 分别是标签和特征向量的平均值。把 $b^*$ 代回去再对 $\boldsymbol{w}$ 求偏导，可以解出

$$ \boldsymbol{w}^* = \big((\boldsymbol{X}-\bar{\boldsymbol{x}}^\top)^\top(\boldsymbol{X}-\bar{\boldsymbol{x}}^\top)\big)^{-1}(\boldsymbol{X}-\bar{\boldsymbol{x}}^\top)^\top(\boldsymbol{y}-\bar y). $$

若加上 $\ell_2$ 正则化（结构风险），最优解变为

$$ \boldsymbol{w}^* = \big((\boldsymbol{X}-\bar{\boldsymbol{x}}^\top)^\top(\boldsymbol{X}-\bar{\boldsymbol{x}}^\top)+\lambda \boldsymbol{I}\big)^{-1}(\boldsymbol{X}-\bar{\boldsymbol{x}}^\top)^\top(\boldsymbol{y}-\bar y). $$

下面把 `optimizer_lsm` 用 PyTorch 实现出来。


In [ ]:
def optimizer_lsm(model, X, y, reg_lambda=0.0):
    """最小二乘法（可带 L2 正则化）求解 Linear 算子的参数。"""
    N, D = X.shape

    # 特征均值（行向量）与标签均值（标量）
    x_bar = X.mean(dim=0, keepdim=True)        # [1, D]
    y_bar = y.mean()

    x_sub = X - x_bar                          # 广播：每行减去均值
    if torch.all(x_sub == 0):
        model.params['b'] = y_bar.unsqueeze(0)
        model.params['w'] = torch.zeros(D, 1)
        return model

    # (X' X + lam I)^-1 X' (y - y_bar)
    A = x_sub.T @ x_sub + reg_lambda * torch.eye(D)
    rhs = x_sub.T @ (y - y_bar)
    w = torch.linalg.solve(A, rhs)
    b = y_bar - x_bar @ w                       # [1, 1]

    model.params['w'] = w                       # [D, 1]
    model.params['b'] = b.squeeze(0)            # [1]
    return model


### 1.5 模型训练 + 评价

在准备了数据、模型、损失和优化器后，我们开始训练。对线性回归而言，最小二乘法一步就能解出最优参数；模型的评价指标和损失函数一致，都用 MSE。


In [ ]:
model = Linear(input_size=1)
optimizer_lsm(model, X_train, y_train)

print(f"w_pred: {model.params['w'].item():.4f}   b_pred: {model.params['b'].item():.4f}")

y_train_pred = model(X_train)
train_error  = mean_squared_error(y_train, y_train_pred).item()
y_test_pred  = model(X_test)
test_error   = mean_squared_error(y_test,  y_test_pred).item()
print(f'train error: {train_error:.4f}')
print(f'test  error: {test_error:.4f}')


从输出结果看，预测出来的 $w$ 与真值 $1.2$ 比较接近，但仍有一定差距，这是噪声引起的。如果把训练样本从 100 增加到 5000，估计误差会进一步减小——这是"更多数据 → 估计方差更小"的最直接体现。


In [ ]:
model_large = Linear(input_size=1)
optimizer_lsm(model_large, X_large, y_large)

print(f"100 sample : w={model.params['w'].item():.4f}  b={model.params['b'].item():.4f}")
print(f"5000 sample: w={model_large.params['w'].item():.4f}  b={model_large.params['b'].item():.4f}")


### 1.6 对照：`nn.Linear` + SGD

实际开发中我们更常用 PyTorch 内置的 `nn.Linear` + 优化器自动求解。这一步用 SGD 训练同一个数据集，验证它能收敛到与最小二乘法相同的参数。


In [ ]:
torch.manual_seed(0)
model_sgd = nn.Linear(1, 1)
opt = optim.SGD(model_sgd.parameters(), lr=0.005)
for _ in range(2000):
    loss = nn.functional.mse_loss(model_sgd(X_train), y_train)
    opt.zero_grad(); loss.backward(); opt.step()
print(f'SGD: w={model_sgd.weight.item():.4f}  b={model_sgd.bias.item():.4f}')


## 2. 多项式回归：模型容量与欠/过拟合

多项式回归是回归任务的一种形式，使用 $M$ 次多项式来建模输入和输出之间的关系：

$$ f(\boldsymbol{x};\boldsymbol{w},b)=w_1 x + w_2 x^2 + \cdots + w_M x^M + b = \boldsymbol{w}^\top \phi(x) + b, $$

其中 $\phi(x)= [x, x^2, \ldots, x^M]^\top$ 称为**多项式基函数**（polynomial basis function），$M$ 是多项式的阶数。从公式可以看出：**多项式回归 = 用基函数把 $x$ 变换到高维空间 + 在变换后的特征上做线性回归**。这也是"线性方法做非线性"的最经典例子。

> 一般地，当输入和输出之间并不是线性关系时，可以定义**非线性基函数**对特征进行变换，使线性回归算法实现非线性曲线拟合。

下面在 $y = \sin(2\pi x) + \epsilon$ 的合成数据上演示，训练集只有 15 个点。我们把这个数据集叫 **ToySin25**：训练集 15 个，测试集 10 个。


In [ ]:
torch.manual_seed(0)


def sin_data(n, noise=0.5):
    x = torch.rand(n, 1)
    y = torch.sin(2 * math.pi * x) + noise * torch.randn_like(x)
    return x, y


X_train, y_train = sin_data(15)
X_test,  y_test  = sin_data(10)

# 真实函数曲线（无噪声），用于可视化对比
x_under = torch.linspace(0, 1, 200).unsqueeze(1)
y_under = torch.sin(2 * math.pi * x_under)

plt.scatter(X_train.numpy(), y_train.numpy(), s=30, label='train')
plt.plot(x_under.numpy(), y_under.numpy(), 'k-', lw=1, label=r'$\sin(2\pi x)$')
plt.legend(); plt.title('ToySin25'); plt.show()


### 2.1 多项式基函数 + 不同阶数拟合

实现一个把 $x$ 变换为 $[x, x^2, \ldots, x^M]$ 的函数 `poly_basis`。然后复用最小二乘法 `optimizer_lsm`：把基函数变换后的高维特征当作普通线性回归来求解就行。


In [ ]:
def poly_basis(x, degree):
    """[N, 1] -> [N, degree]：列依次为 x, x^2, …, x^M。
    degree=0 时返回空列；偏置项由 fit_predict 里独立拼上去。"""
    if degree == 0:
        return torch.empty(x.shape[0], 0)
    return torch.cat([x ** k for k in range(1, degree + 1)], dim=1)


def fit_predict(X_train, y_train, X_eval, degree, lam=0.0):
    """多项式基 + （带正则的）闭式最小二乘拟合，再在 X_eval 上预测。"""
    Phi_t = poly_basis(X_train, degree)
    Phi_e = poly_basis(X_eval,  degree)
    Phi_t1 = torch.cat([Phi_t, torch.ones_like(X_train[:, :1])], dim=1)
    Phi_e1 = torch.cat([Phi_e, torch.ones_like(X_eval[:,  :1])], dim=1)
    D = Phi_t1.shape[1]
    I = torch.eye(D); I[-1, -1] = 0    # 偏置不参与正则化
    A = Phi_t1.T @ Phi_t1 + lam * I
    w = torch.linalg.solve(A, Phi_t1.T @ y_train)
    return Phi_e1 @ w


fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, M in zip(axes.flat, [0, 1, 3, 8]):
    y_hat = fit_predict(X_train, y_train, x_under, degree=M)
    ax.scatter(X_train.numpy(), y_train.numpy(), s=20)
    ax.plot(x_under.numpy(), y_under.numpy(), 'k-', lw=1, label=r'$\sin(2\pi x)$')
    ax.plot(x_under.numpy(), y_hat.numpy(),  'r-', lw=1.5, label=f'M={M} fit')
    ax.set_ylim(-2, 2); ax.set_title(f'M={M}'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


**观察**：

- $M=0$ 或 $M=1$：表示能力不足，**欠拟合**——曲线甚至跟数据形状都对不上；
- $M=3$：模型容量与数据匹配，已经能很好地近似 $\sin$；
- $M=8$：表示能力过强但训练数据只有 15 个点，**过拟合**——曲线在数据稀疏处剧烈波动。


### 2.2 训练 / 测试误差随阶数变化

把上面的过程对 $M=0,\ldots,8$ 都跑一遍，看训练集和测试集误差如何随模型容量变化。


In [ ]:
def mse(y_hat, y):
    return ((y_hat - y) ** 2).mean().item()


train_errs, test_errs = [], []
for M in range(9):
    y_train_pred = fit_predict(X_train, y_train, X_train, degree=M)
    y_test_pred  = fit_predict(X_train, y_train, X_test,  degree=M)
    train_errs.append(mse(y_train_pred, y_train))
    test_errs.append(mse(y_test_pred,   y_test))

plt.plot(train_errs, '-o', label='train')
plt.plot(test_errs,  '-s', label='test')
plt.xlabel('polynomial degree M'); plt.ylabel('MSE')
plt.yscale('log'); plt.grid(alpha=0.3); plt.legend(); plt.show()
print('train MSE per M:', [round(e, 4) for e in train_errs])
print('test  MSE per M:', [round(e, 4) for e in test_errs])


**典型现象**：

1. 阶数较低（$0 \sim 2$）时模型欠拟合，训练和测试误差都很高；
2. 随着阶数增加（$3 \sim 5$），模型表示能力增强，训练误差继续降低；
3. 阶数继续增加（$6 \sim 8$）后，虽然模型表示能力强，但**把训练数据中的噪声也当成信号学进去了**——训练误差继续降低，测试误差显著升高，模型过拟合。


### 2.3 用 $\ell_2$ 正则化压住过拟合

对模型过拟合的情况，可以通过往误差函数中添加一个**惩罚项**来避免参数倾向于较大取值。下面在 $M=8$ 上加 $\ell_2$ 正则化，对比正则与不正则的拟合效果。

ridge 形式的闭式解：

$$ \hat{\boldsymbol{w}} = (\boldsymbol{X}^\top \boldsymbol{X} + \lambda \boldsymbol{I})^{-1} \boldsymbol{X}^\top \boldsymbol{y}. $$

**注意**：偏置项通常不正则化（在我们 `fit_predict` 里通过 `I[-1, -1] = 0` 实现）。

下面用**真实函数曲线 $\sin(2\pi x)$ 上密集采样**作为评估面（而不是用噪声测试集），这样能直接看到 $\ell_2$ 让拟合曲线**更贴近真分布**的效果。


In [ ]:
M, lam = 8, 0.1
y0 = fit_predict(X_train, y_train, x_under, degree=M, lam=0.0)
yr = fit_predict(X_train, y_train, x_under, degree=M, lam=lam)

# 在干净的 (x_under, y_under) 上算 MSE，反映"拟合曲线 vs 真分布"的距离
print(f'MSE on true sin surface  no-reg  M=8     :  {mse(y0, y_under):.4f}')
print(f'MSE on true sin surface  L2     λ={lam}  :  {mse(yr, y_under):.4f}')

plt.scatter(X_train.numpy(), y_train.numpy(), s=20, label='train')
plt.plot(x_under.numpy(), y_under.numpy(), 'k-',  lw=1,   label=r'$\sin(2\pi x)$')
plt.plot(x_under.numpy(), y0.numpy(),      'r--', lw=1.2, label='M=8, no reg')
plt.plot(x_under.numpy(), yr.numpy(),      'b-',  lw=1.5, label=f'M=8, L2 (λ={lam})')
plt.ylim(-2, 2); plt.legend(); plt.show()


**\\\\想一想\\\\**：给定一个现实场景任务，我们可以有两种选择：

1. 使用复杂的模型，并增大正则化系数来避免过拟合；
2. 直接使用简单模型。

哪种选择更好？答案与数据量、特征复杂度、可解释性需求都有关。

**\\\\练一练\\\\**：调整训练数据的样本数量，观察不同模型复杂度下、避免过拟合需要多少样本。


## 小结

本节按机器学习实践的**五要素**走了一遍：

- **数据**：在 ToyLinear150 和 ToySin25 上演示数据采样与训练/测试划分；
- **模型**：自定义 `Op` 接口的 `Linear` 算子，也用 `nn.Linear` 做对照；
- **学习准则**：均方误差，含经验风险与结构风险（加 $\ell_2$）；
- **优化算法**：最小二乘法直接解析解 + SGD 梯度法验证；
- **评价指标**：训练/测试 MSE。

关键工具速查：

- **闭式解**：`torch.linalg.lstsq` 求无正则的最小二乘；带 $\ell_2$ 正则用 `torch.linalg.solve` 解 $(X^\top X + \lambda I) w = X^\top y$；
- **多项式回归 = 基函数变换 + 线性回归**：模型容量随阶数 $M$ 增长；$M$ 过大 + 数据少 → 过拟合；
- **$\ell_2$ 正则**压低系数模长，曲线更平滑。PyTorch 训练时更常用 `torch.optim.*` 的 `weight_decay` 参数实现等效效果。

下一节用 `Runner` 类把"数据 → 模型 → 训练 → 评估 → 保存"封装成一个可复用骨架，并跑通加州房价回归案例。
